## Group Convolutional Neural Networks

[open on colab](https://drive.google.com/file/d/1Q2Si9ScwoBgCR9Fq3Jn1i0Ht0F5pzIWb/view?usp=sharing)

Before I implement the paper [Group Equivariant CNNs](https://arxiv.org/abs/1602.07576) by Taco and Max. I must first explain Equivariance and Invariance:

* Equivariance

    A desirable characteristic when approximating anything (e.g. a cat image) using neural networks is having it be equivariant to the input. No matter how the desired object moves inside the input you want the activation to be where the object is

* Invariance

    A function is said to be invariant when the same input goes to the function no matter what transformation happened to it, the function output will be the same.


<img src="https://i.imgur.com/UCZYCXD.png" width="800"/>


## **Why equivariance matters?**

- No info is lost when the input is transformed

- Guaranteed stability to (local + global) transformations

- In traditional CNN classifiers it also helps an important component!

<img src="https://i.imgur.com/NG79MlG.jpeg" width="800"/>

An image to demonstrate the difference with the mathematical definition of both equivariance and invariance:

<img src="https://i.imgur.com/2ta4jrn.jpeg" width="800"/>

<img src="https://i.imgur.com/GxRynB3.png" width="800"/>

[source of the image](https://www.youtube.com/watch?v=03MbWVlbefM&t=1393s)

Important to note that both equivariance and invariance are always defined with respect to a set of transformation class. In classical CNNs it is translations in 2D space.

Classical CNNs are translation equivariant, meaning that when an object in the input is translated, the feature maps translate correspondingly. or stay constant for invariance. Which are the pooling layers represent invariant component in the network.

<img src="https://i.imgur.com/NT2C5Fl.png" width="800"/>


<img src="https://i.imgur.com/oTeUSFW.png" width="800"/>


## **Why's it important to change inductive biases?**


<img src="https://i.imgur.com/ZXMbBx6.png" width="800"/>


---

**FYI:**

* **Classical CNNs** are only equivariant to translations, but not rotations or reflections.

* **MLPs** are not equivariant to any spatial transformation; the input structure matters.

* **Transformers** without positional encoding are permutation invariance.

## How are they perm invar:

<img src="https://i.imgur.com/oCnuLd0.png" width="800"/>


## An application:

<img src="https://i.imgur.com/mxBr1xF.png" width="800"/>

---

## **What if we instead of translation had rotation/scaling/shearing/permutation?**
### **What would you do?**

<img src="https://i.imgur.com/TbPsFIW.jpeg" width="800"/>


## **Simple Solution is to apply augs:**

<img src="https://i.imgur.com/Br6hC0S.png" width="800"/>

### Downsides:

- no guarantee it will be invariant

- only applies to the input layer

- valuable net capacity spent on learning invariance

- redundancy in feature representation

### **Proof?**
Read:

<img src="https://i.imgur.com/trbXsys.png" width="800"/>

[Naturally occuring equivariance](https://distill.pub/2020/circuits/equivariance/)

---

## **Group theory**


<img src="https://i.imgur.com/p757GAK.png" width="800"/>


<img src="https://i.imgur.com/GTmuN8Y.png" width="800"/>


### **Is this a group?**

<img src="https://i.imgur.com/mHChyw5.png" width="800"/>

### **Apply one step into cayley table**

<img src="https://i.imgur.com/6RkdK0C.png" width="800"/>


<img src="https://i.imgur.com/5j9yBOE.png" width="800"/>

<img src="https://i.imgur.com/nEK95YA.png" width="800"/>


As you can see, the convolution formula applies a kernel $k$ to the signal $f$ (which in our case its an image), which can be expressed as 2 summations on the right hand side of the equation, we can see that the kernel $k_i$ shifted across the input by $(x-y)$ and this is what gives the classical CNNs translation equivariance!

The key idea behind Group Equivariant CNNs (G-CNNs) is to extend this equivariance from just translations to larger transformation groups (e.g., rotations, reflections). By doing so, the network respects the symmetries of the data and generalizes better with fewer parameters.
But this typically works with discrete groups like the cyclic group $C_4$ $=$ {$0°, 90°, 180°, 270°$}.

I highly suggest to checkout this [video on equivariant neural nets](https://www.youtube.com/watch?v=2bP_KuBrXSc) which will help you alot to grasp the basics of group theory and symmetries in math.

I will build the simplest G-CNN which only consider the rotations of cyclic group $C_4$.

## Visualization of conventional vs group CNNs in activation response to rotation

<img src="https://i.imgur.com/PCC5442.gif" width="800"/>

<img src="https://i.imgur.com/KU85EPi.gif" width="800"/>




### Reqs

In [ ]:
#@title deps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim

from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

### Helper

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:

import matplotlib.pyplot as plt
import random

#@title showing preds vs actual in rotation

import matplotlib.pyplot as plt

def show_predictions(model, dataset, device, indices, title_prefix=""):
    model.eval()

    images = torch.stack([dataset[i][0] for i in indices]).to(device)
    labels = torch.tensor([dataset[i][1] for i in indices]).to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = outputs.argmax(dim=1)

    n = len(indices)
    cols = 6
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.2))
    axes = axes.flatten() if n > 1 else [axes]

    for i in range(n):
        img = images[i].cpu().squeeze().numpy()
        pred = preds[i].item()
        true = labels[i].item()
        correct = pred == true

        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(f"Pred: {pred} | True: {true}",
                           color="green" if correct else "red", fontsize=10)
        axes[i].axis("off")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title_prefix, fontsize=13)
    plt.tight_layout()
    plt.show()

def show_feature_maps(model, x, model_name="G-CNN"):
    model.eval()
    with torch.no_grad():
        feats = model(x, return_features=True)

    if model_name == "G-CNN":
        # feats shape: [B, 4, 8, H, W]
        feats = feats[0]  # first image in batch → [4, 8, H, W]
        group_rot_names = ["0°", "90°", "180°", "270°"]
        for g in range(4):
            fig, axs = plt.subplots(1, 8, figsize=(16, 2))
            for i in range(8):
                axs[i].imshow(feats[g, i].cpu(), cmap='viridis')
                axs[i].axis('off')
                axs[i].set_title(f'F{i}')
            fig.suptitle(f'G-CNN - Group Element: {group_rot_names[g]}', fontsize=14)
            plt.show()

    elif model_name == "CNN":
        # feats shape: [B, 8, H, W]
        feats = feats[0]  # [8, H, W]
        fig, axs = plt.subplots(1, 8, figsize=(16, 2))
        for i in range(8):
            axs[i].imshow(feats[i].cpu(), cmap='viridis')
            axs[i].axis('off')
            axs[i].set_title(f'F{i}')
        fig.suptitle('Standard CNN - Feature Maps', fontsize=14)
        plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=range(10), yticklabels=range(10))
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

In [ ]:
def get_all_preds(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)

In [ ]:
# Training function
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
#@title dataset stats + image grid visualization

def show_dataset_stats(loader, name="Dataset"):
    all_labels = []
    batch_x, batch_y = next(iter(loader))

    for _, y in loader:
        all_labels.extend(y.numpy())
    all_labels = np.array(all_labels)

    print(f"--- {name} ---")
    print(f"Num samples:     {len(all_labels)}")
    print(f"Num batches:     {len(loader)}")
    print(f"Batch shape:     {tuple(batch_x.shape)}  (B, C, H, W)")
    print(f"Pixel range:     [{batch_x.min().item():.3f}, {batch_x.max().item():.3f}]")
    print(f"Pixel mean/std:  {batch_x.mean().item():.4f} / {batch_x.std().item():.4f}")

    unique, counts = np.unique(all_labels, return_counts=True)
    print(f"Class counts:    {dict(zip(unique.tolist(), counts.tolist()))}")

    plt.figure(figsize=(6, 3))
    plt.bar(unique, counts)
    plt.xticks(unique)
    plt.xlabel("Digit")
    plt.ylabel("Count")
    plt.title(f"{name} — Class Distribution")
    plt.show()


def show_image_grid(loader, n=8, title="Sample Images"):
    x, y = next(iter(loader))
    x, y = x[:n], y[:n]

    fig, axs = plt.subplots(1, n, figsize=(2 * n, 2))
    for i in range(n):
        axs[i].imshow(x[i].squeeze(0), cmap='gray')
        axs[i].axis('off')
        axs[i].set_title(str(y[i].item()))
    fig.suptitle(title, fontsize=14)
    plt.show()

### Data

In [ ]:
#@title creation of rotation augmentation

# Random C4 rotation: 0°, 90°, 180°, 270° — used only for the "CNN + augmentation" model
def random_c4_rotation(img):
    k = torch.randint(0, 4, ()).item()
    return transforms.functional.rotate(img, angle=90 * k)

# --- Transforms ---
# Plain CNN baseline: NO rotation augmentation at train time
train_transform_plain = transforms.Compose([
    transforms.ToTensor()
])

# CNN + augmentation / GCNN input pipeline: random C4 rotation at train time
train_transform_aug = transforms.Compose([
    transforms.Lambda(random_c4_rotation),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.ToTensor()
])

# --- Base datasets (unrotated, raw) ---
train_dataset_plain = datasets.MNIST(root='./data', train=True, transform=train_transform_plain, download=True)
train_dataset_aug   = datasets.MNIST(root='./data', train=True, transform=train_transform_aug,   download=True)
test_dataset_original = datasets.MNIST(root='./data', train=False, transform=test_transform, download=True)

# --- Deterministic rotated test set ---
# Instead of a random-per-getitem transform (which changes every epoch and makes
# your reported "rotated test accuracy" non-reproducible), we materialize ONE
# fixed random rotation per test image, seeded, so every model is evaluated on
# the exact same rotated test set.
def build_fixed_rotated_dataset(base_dataset, seed=42):
    g = torch.Generator().manual_seed(seed)
    fixed_ks = torch.randint(0, 4, (len(base_dataset),), generator=g)

    rotated_images = []
    labels = []
    for i in range(len(base_dataset)):
        img, y = base_dataset[i]  # img is already a raw PIL image if we bypass ToTensor here
        k = fixed_ks[i].item()
        img = transforms.functional.rotate(img, angle=90 * k)
        rotated_images.append(transforms.functional.to_tensor(img))
        labels.append(y)

    images_tensor = torch.stack(rotated_images)
    labels_tensor = torch.tensor(labels)
    return torch.utils.data.TensorDataset(images_tensor, labels_tensor)

# Need the raw PIL version (no ToTensor) to rotate cleanly before building the fixed set
test_dataset_raw = datasets.MNIST(root='./data', train=False, transform=None, download=True)
test_dataset_rotated = build_fixed_rotated_dataset(test_dataset_raw, seed=42)

# --- DataLoaders ---
train_loader_plain = DataLoader(train_dataset_plain, batch_size=64, shuffle=True)
train_loader_aug   = DataLoader(train_dataset_aug,   batch_size=64, shuffle=True)

test_loader_rotated  = DataLoader(test_dataset_rotated,  batch_size=64)
test_loader_original = DataLoader(test_dataset_original, batch_size=64)

In [ ]:
#@title how the data looks

show_dataset_stats(train_loader_plain, name="Train (plain, unrotated)")
show_dataset_stats(train_loader_aug,   name="Train (C4 augmented)")
show_dataset_stats(test_loader_original, name="Test (original)")
show_dataset_stats(test_loader_rotated,  name="Test (fixed rotated)")

show_image_grid(train_loader_plain, n=10, title="Train batch (plain, unrotated)")
show_image_grid(train_loader_aug,   n=10, title="Train batch (C4 augmented)")
show_image_grid(test_loader_original, n=10, title="Test batch (original)")
show_image_grid(test_loader_rotated,  n=10, title="Test batch (fixed rotated)")

### Archeticture

#### Classical CNN for comparison

In [ ]:
class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 28, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(28, 28, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(28, 10)
        )

    def forward(self, x, return_features=False):
        feat = self.features(x)
        if return_features:
            return feat  # [B, 28, 28, 28]
        return self.classifier(feat)

In [ ]:
model_cnn = StandardCNN().to(device)
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
print(f'model size: {sum(p.numel() for p in model_cnn.parameters()):,} parameters')

In [ ]:
for epoch in range(1, 6):
    loss = train(model_cnn, train_loader_plain, optimizer_cnn, criterion)
    acc_rot = evaluate(model_cnn, test_loader_rotated)
    acc_orig = evaluate(model_cnn, test_loader_original)
    print(f"[Standard CNN] Epoch {epoch}: Loss={loss:.4f}, Acc(Rotated)={acc_rot:.4f}, Acc(Original)={acc_orig:.4f}")

In [ ]:
cnn_preds, cnn_labels = get_all_preds(model_cnn, test_loader_rotated)
print("=== Standard CNN Classification Report ===")
print(classification_report(cnn_labels, cnn_preds, digits=4))
plot_confusion_matrix(cnn_labels, cnn_preds, title="Standard CNN Confusion Matrix")

#### Rotation-Augmented-CNN

In [ ]:
#@title Standard CNN + rotation augmentation (baseline w/ data aug)

model_cnn_aug = StandardCNN().to(device)
optimizer_cnn_aug = optim.Adam(model_cnn_aug.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
print(f'model size: {sum(p.numel() for p in model_cnn_aug.parameters()):,} parameters')

for epoch in range(1, 6):
    loss = train(model_cnn_aug, train_loader_aug, optimizer_cnn_aug, criterion)
    acc_rot = evaluate(model_cnn_aug, test_loader_rotated)
    acc_orig = evaluate(model_cnn_aug, test_loader_original)
    print(f"[CNN + Aug] Epoch {epoch}: Loss={loss:.4f}, Acc(Rotated)={acc_rot:.4f}, Acc(Original)={acc_orig:.4f}")

In [ ]:
cnn_aug_preds, cnn_aug_labels = get_all_preds(model_cnn_aug, test_loader_rotated)
print("=== CNN + Augmentation Classification Report ===")
print(classification_report(cnn_aug_labels, cnn_aug_preds, digits=4))
plot_confusion_matrix(cnn_aug_labels, cnn_aug_preds, title="CNN + Augmentation Confusion Matrix")

#### G-CNN

In [ ]:
def rotate_filter_90s(w, k):
    return torch.rot90(w, k=k, dims=[-2, -1])


class GCNN_C4_Layer(nn.Module):
    """Lifting layer: Z2 -> p4"""
    def __init__(self, in_channels, out_channels, kernel_size, padding=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size)
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x):
        outputs = []
        for k in range(4):
            w_rot = rotate_filter_90s(self.weight, k)
            out = F.conv2d(x, w_rot, bias=self.bias, padding=self.kernel_size // 2)
            outputs.append(out)
        return torch.stack(outputs, dim=1)


class GConvC4Layer(nn.Module):
    """Group conv: p4 -> p4"""
    def __init__(self, in_channels, out_channels, kernel_size, padding=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.weight = nn.Parameter(
            torch.randn(4, out_channels, in_channels, kernel_size, kernel_size)
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x):
        outputs = []
        for g in range(4):
            acc = 0
            for h in range(4):
                idx = (h - g) % 4
                w = rotate_filter_90s(self.weight[idx], g)
                acc = acc + F.conv2d(x[:, h], w, padding=self.kernel_size // 2)
            acc = acc + self.bias.view(1, -1, 1, 1)
            outputs.append(acc)
        return torch.stack(outputs, dim=1)


class GroupPool(nn.Module):
    def forward(self, x):
        return x.mean(dim=1)


class SimpleGCNN_C4(nn.Module):
    def __init__(self):
        super().__init__()
        self.gconv1 = GCNN_C4_Layer(1, 12, kernel_size=3)     # C1 = 12
        self.relu1 = nn.ReLU()
        self.gconv2 = GConvC4Layer(12, 17, kernel_size=3)     # C2 = 17
        self.relu2 = nn.ReLU()
        self.pool = GroupPool()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(17, 10)
        )

    def forward(self, x, return_features=False):
        x = self.relu1(self.gconv1(x))
        x = self.relu2(self.gconv2(x))
        if return_features:
            return x
        x = self.pool(x)
        return self.classifier(x)


In [ ]:
x = torch.randn(2, 1, 28, 28)
model_gcnn = SimpleGCNN_C4()
out = model_gcnn(x)
model_gcnn = model_gcnn.to(device)
print(out.shape)
print(f'model size: {sum(p.numel() for p in model_gcnn.parameters()):,} parameters')

### Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer_gcnn = optim.Adam(model_gcnn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(1, 6):
    loss = train(model_gcnn, train_loader_plain, optimizer_gcnn, criterion)
    acc_rot = evaluate(model_gcnn, test_loader_rotated)
    acc_orig = evaluate(model_gcnn, test_loader_original)
    print(f"[G-CNN] Epoch {epoch}: Loss={loss:.4f}, Acc(Rotated)={acc_rot:.4f}, Acc(Original)={acc_orig:.4f}")

### Eval

In [ ]:
gcnn_preds, gcnn_labels = get_all_preds(model_gcnn, test_loader_rotated)
print("=== G-CNN Classification Report ===")
print(classification_report(gcnn_labels, gcnn_preds, digits=4))
plot_confusion_matrix(gcnn_labels, gcnn_preds, title="G-CNN Confusion Matrix")

In [ ]:
#@title showing preds vs actual in rotation
# G CNN on rotated digits, for comparison
my_indices = [0, 17, 5, 103, 108, 252]

show_predictions(model_cnn, test_loader_rotated.dataset, device, my_indices,
                  title_prefix="Standard CNN on Rotated MNIST")
show_predictions(model_cnn_aug, test_loader_rotated.dataset, device, my_indices,
                  title_prefix="Augmentation CNN on Rotated MNIST")
show_predictions(model_gcnn, test_loader_rotated.dataset, device, my_indices,
                  title_prefix="G-CNN on Rotated MNIST")


## Home exercise:

Build your own 4-cyclic group of transformations (Shearing/Scaling/etc...) and verify the group via Cayley-Table

Build a G-CNN that have this custom inductive bias and apply this augmentation on the test MNIST only, train your G-CNN on normal MNIST and see if it doesn't

In [ ]:
## Write your transformation here
def transformation(w, k):
    return


## Define the kernel transformation layer
class GCNN_C4_Layer(nn.Module):
    """Lifting layer: Z2 -> transformation space"""
    def __init__(self, in_channels, out_channels, kernel_size, padding=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size)
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x):
        outputs = []
        for k in range(4):
            w_rot = transformation(self.weight, k)
            out = F.conv2d(x, w_rot, bias=self.bias, padding=self.kernel_size // 2)
            outputs.append(out)
        return torch.stack(outputs, dim=1)

## Write the sliding convolution layer (how the group operator kernel should slide)
class GConvC4Layer(nn.Module):
    """Group conv:  transformation space -> same space """
    def __init__(self, in_channels, out_channels, kernel_size, padding=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.weight = nn.Parameter(
            torch.randn(4, out_channels, in_channels, kernel_size, kernel_size)
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x):
        outputs = []
        for g in range(4):

        return torch.stack(outputs, dim=1)


class GroupPool(nn.Module):
    def forward(self, x):
        return x.mean(dim=1)


class SimpleGCNN_C4(nn.Module):
    def __init__(self):
        super().__init__()
        self.gconv1 = GCNN_C4_Layer(1, 12, kernel_size=3)     # C1 = 12
        self.relu1 = nn.ReLU()
        self.gconv2 = GConvC4Layer(12, 17, kernel_size=3)     # C2 = 17
        self.relu2 = nn.ReLU()
        self.pool = GroupPool()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(17, 10)
        )

    def forward(self, x, return_features=False):
        x = self.relu1(self.gconv1(x))
        x = self.relu2(self.gconv2(x))
        if return_features:
            return x
        x = self.pool(x)
        return self.classifier(x)


## Real-world applications of modified inductive-biases

<img src="https://i.imgur.com/6wjZxeW.png" width="800"/>

[SE(3) Transformer for 3D point clouds and graphs](https://distill.pub/2020/circuits/equivariance/)


<img src="https://i.imgur.com/r5rPOAp.png" width="800"/>

[Rotation Equivariant CNNs for Digital Pathology (P4 G-CNNs)](https://arxiv.org/abs/1806.03962)

<img src="https://i.imgur.com/Z6sm4hg.png" width="800"/>

[Geometry-complete diffusion for 3D molecule generation and optimization](https://www.nature.com/articles/s42004-024-01233-z)

## End of geometry

<img src="https://i.imgur.com/X2adWrk.jpeg" width="400"/>


Done by Anas Aldadi



